# Remove buffer flush indicators in raw data of mass time series

This Colab identifies and treats the interruptions caused by buffer flushes on the SD card. Every time the STUAART saves data on the SD card, a string such as "--, --, --, -- \n" is saved to indicate the moment the buffer flushed its content on the SD card. We need to remove it to display the mass time series properly.

Additionnaly, mass data are sometimes not in the right file. For example, data acquired at 23:59:59 at day 1 might be in the file of day 2. This Colab ensures that the saved files have the correct data for each day.

The cleaned mass time series, without any "--" interrupting the floats, are saved in a new folder called *WithoutBufferFlushes* where is the raw data.

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import os
import shutil

## Step 1 : Define variables and mount the drive

Define variables :
 - You must change *project_directory* and *data_path* to where your .csv files are saved in your Google drive.
 - *debug_mode* is True if you want to print the definition of some variables.


The secon cell is used to mount your Google Drive. To access the data on your Google drive, it must be mounted. When you first run the script, you must follow the steps and accept access to your drive.

In [2]:
# Change these variables
project_directory = "/content/drive/My Drive/DCCM_Lab/labdata/vpineaunoel/stuaart"
data_path = project_directory + "/20260629-3miceInSTUAARTs_2/Red/"
debug_mode = False
file_size_threshold = 200 * 1024 * 1024

In [3]:
# Do not change this section of the script
from google.colab import drive
drive.mount('/content/drive/')
import os
os.chdir(project_directory)
local_temp_folder = "/content/temp_csv_data"

MessageError: Error: credential propagation was unsuccessful

In [ ]:
def create_temporary_local_folder(local_temp_folder = "/content/temp_csv_data"):
  # Create a temporary local folder for the heavy files
  os.makedirs(local_temp_folder, exist_ok=True)

def identify_buffer_flushes(time, data, debug_mode, remove=False):
  indices_of_buffer_flushes = np.where(time == "--")[0]
  valid_mask = (time != "--")
  modified_time = time.copy()
  for i in np.where(valid_mask)[0]:
    modified_time[i] = float(time_to_float(modified_time[i]))

  if remove:
    mass_data_without_flushes = np.delete(data, indices_of_buffer_flushes, axis=0)
    time_without_flushes = np.delete(modified_time, indices_of_buffer_flushes)
    return time_without_flushes.astype(float), mass_data_without_flushes.astype(float)
  else:
    return modified_time, data


def time_to_float(time_str):
    """Converts a single valid 'HH:MM:SS.milliseconds' string to float hours."""
    parts = time_str.split(':')
    return float(parts[0]) + float(parts[1]) / 60.0 + float(parts[2]) / 3600.0


In [ ]:
path = Path(data_path)
new_path = path / "WithoutBufferFlushes"
new_path.mkdir(parents=True, exist_ok=True)

# Create temporary local folder for the heavy files, if there is any
create_temporary_local_folder(local_temp_folder=local_temp_folder)

files = sorted(os.listdir(data_path))
files_to_keep = []
if debug_mode:
  print("All files identified : ", files)
for file in files: # remove files/folders that are not mass time series
  if debug_mode:
    print(file)
  if file[-4:] != ".csv":
    if debug_mode:
      print("Removing file : ", file)
  else:
    files_to_keep.append(file)
if debug_mode:
  print("Files kept : ", files_to_keep)

all_data = []
all_time = []
print("Accessing raw mass time series : ")
for file in tqdm(files_to_keep):
  if debug_mode:
    print(file)

  file_size = os.path.getsize(data_path + file)
  if file_size > file_size_threshold:
    if debug_mode:
      print(f"File size is bigger than {file_size_threshold/ 1e6:.1f} MB. Creating local copy of the files to {local_temp_folder}.")
    local_path = os.path.join(local_temp_folder, file)
    if not os.path.exists(local_path):
      shutil.copy(data_path + file, local_path)
    read_path = local_path
  else:
    read_path = data_path + file

  data = np.array(pd.read_csv(read_path))
  time = data[:,0]
  mass_data = data[:, 1:]
  if debug_mode:
    print("Raw Time : ", time, time.shape, time.dtype)
    print("Raw Mass Data : ", mass_data, mass_data.shape, mass_data.dtype)
  all_data.append(mass_data)
  all_time.append(time)

concatenated_mass_data = np.concatenate(all_data, axis=0)
concatenated_time = np.concatenate(all_time)

print("Removing buffer flush indicators.")
concatenated_time, concatenated_mass_data = identify_buffer_flushes(time=concatenated_time, data=concatenated_mass_data, debug_mode=debug_mode, remove=True)
if debug_mode:
  print("CONCATENATED TIME : ", concatenated_time.shape)
  print("CONCATENATED MASS DATA : ", concatenated_mass_data.shape)

diffs = np.diff(concatenated_time)
rollover_next_day = diffs < -12.0
indices_of_next_day = np.where(rollover_next_day)[0]
indices_of_beginning_and_end_days = np.insert(indices_of_next_day, 0, 0)
indices_of_beginning_and_end_days = np.append(indices_of_beginning_and_end_days, concatenated_time.shape[0])

print("Saving mass time series per day : ")
for i in tqdm(range(indices_of_beginning_and_end_days.shape[0]-1)):
  start = indices_of_beginning_and_end_days[i]
  end = indices_of_beginning_and_end_days[i+1]
  length = end - start
  data_without_buffer_flushes = np.zeros(shape=(length, concatenated_mass_data.shape[1]+1))
  data_without_buffer_flushes[:, 0] = concatenated_time[start:end]
  data_without_buffer_flushes[:, 1:] = concatenated_mass_data[start:end, :]
  df = pd.DataFrame(data_without_buffer_flushes)
  if debug_mode:
    print("Dataframe : ",  df)
  df.to_csv(new_path / (files_to_keep[i][:-4] + "-WithoutBufferFlushes.csv"), index=False)

# Clean up the local file to prevent filling up Colab's disk storage
if local_path and os.path.exists(local_path):
  os.remove(local_path)